# 3-7 merge连接操作

`merge()` 根据共同的键，将两张表的字段连接到一起。理解连接前，先确认匹配字段和需要保留哪些记录。

> 约定：使用 `pd` 作为 Pandas 的别名，使用 `np` 作为 NumPy 的别名。本节可独立从上到下运行。

In [ ]:
import pandas as pd
import numpy as np

## 1. 准备两张表

左表的连接键是 `lkey`，右表是 `rkey`。两表都有 `value`，连接后用后缀区分来源。

In [ ]:
left = pd.DataFrame({"lkey": ["foo", "bar", "baz", "lemon"], "value": [1, 2, 3, 5]})
right = pd.DataFrame({"rkey": ["foo", "bar", "baz", "jack"], "value": [5, 6, 7, 8]})
left

In [ ]:
right

## 2. 四种连接方式

| `how` | 名称 | 保留的键 | 本例行数 |
|---|---|---|---|
| `inner` | 内连接 | 两表都有的键 | 3 |
| `left` | 左连接 | 左表的所有键 | 4 |
| `right` | 右连接 | 右表的所有键 | 4 |
| `outer` | 外连接 | 两表出现过的所有键 | 5 |

行数是本例键不重复时的结果。无法匹配的字段用缺失值表示。

### 2.1 内连接

只保留 `foo`、`bar`、`baz`。`suffixes` 按“左表、右表”的顺序设置同名列后缀。

In [ ]:
pd.merge(left, right, left_on="lkey", right_on="rkey", how="inner", suffixes=("_left", "_right"))

### 2.2 左连接

左表 `lemon` 仍然保留，但右表对应字段为空。

In [ ]:
pd.merge(left, right, left_on="lkey", right_on="rkey", how="left", suffixes=("_left", "_right"))

### 2.3 右连接

右表 `jack` 仍然保留，但左表对应字段为空。

In [ ]:
pd.merge(left, right, left_on="lkey", right_on="rkey", how="right", suffixes=("_left", "_right"))

### 2.4 外连接

保留所有键，包括只出现在某一张表中的键。

In [ ]:
pd.merge(left, right, left_on="lkey", right_on="rkey", how="outer", suffixes=("_left", "_right"))

## 3. 常用参数与检查方法

| 参数 | 用法 |
|---|---|
| `on` | 两表的连接键同名时使用 |
| `left_on` / `right_on` | 两表的连接键不同名时分别指定 |
| `suffixes` | 给非连接键的重名列增加后缀 |
| `indicator=True` | 增加 `_merge` 列，显示记录来源 |
| `validate` | 检查键是一对一、一对多等关系 |

In [ ]:
right_same_key = right.rename(columns={"rkey": "lkey"})
pd.merge(left, right_same_key, on="lkey", how="outer", suffixes=("_left", "_right"), indicator=True, validate="one_to_one")

`_merge` 的 `both` 表示两表都存在，`left_only` / `right_only` 表示只在左表 / 右表中存在。

## 4. 重复键会增加结果行数

一个键在左表出现 2 次、右表出现 3 次时，会产生 `2 × 3 = 6` 行匹配；连接前要检查键是否唯一。

In [ ]:
a = pd.DataFrame({"key": ["A", "A"], "x": [1, 2]})
b = pd.DataFrame({"key": ["A", "A", "A"], "y": [10, 20, 30]})
a.merge(b, on="key", how="inner")

补充：Pandas 的空连接键可能彼此匹配，这与许多 SQL 数据库的行为不同。真实数据连接前应先检查缺失键。

## 5. 要点总结

- `inner` 取共有键，`left` / `right` 保留对应一侧，`outer` 保留全部键。
- 同名连接键用 `on`，不同名连接键用 `left_on` 和 `right_on`。
- 通过 `suffixes` 区分来源，通过 `indicator` 查看匹配情况。
- 重复键可能让结果行数增加，`validate` 可检查预期的连接关系。